**Feature pruning** is the model-free, post-hoc reduction of a ``df_feat`` (here from :meth:`CPP.run` on the gamma-secretase ``DOM_GSEC`` dataset) before model-based selection. :meth:`SequenceFeature.prune_by_variance` drops near-constant features — those whose feature values barely change across the samples — measured on the feature matrix that :meth:`SequenceFeature.feature_matrix` builds from ``df_parts``. Recommended order: **variance -> correlation -> select_features**.

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False

df_seq = aa.load_dataset(name="DOM_GSEC", n=20)
labels = df_seq["label"].to_list()
sf = aa.SequenceFeature()
# gamma-secretase geometry: the TMD (~20 aa) comes from each protein's tmd_start/tmd_stop,
# flanked by short juxtamembrane domains of 4 residues each.
df_parts = sf.get_df_parts(df_seq=df_seq, jmd_n_len=4, jmd_c_len=4)
df_feat = aa.CPP(df_parts=df_parts).run(labels=labels, n_filter=50)
print(f"features from CPP.run: {len(df_feat)}")

CPP using the Python kernel fallback — the compiled Cython extension is not available in this install. Output is bit-exact with the Cython path but ~2x slower. Reinstall via `pip install --force-reinstall aaanalysis` to fetch a prebuilt wheel.


features from CPP.run: 50


With the default ``threshold=0.0`` only strictly constant features are removed (population variance over all samples; ``threshold`` is in variance units). The result is a row-filtered ``df_feat`` with the same schema:

In [2]:
df_var = sf.prune_by_variance(df_feat=df_feat, df_parts=df_parts, threshold=0.0)
print(f"kept {len(df_var)} of {len(df_feat)} features")
aa.display_df(df_var, n_rows=10, show_shape=True)

kept 50 of 50 features
DataFrame shape: (50, 13)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD-Pattern(C,4,8)-BEGF750101",Conformation,α-helix,α-helix,"Conformational ...in-Dirkx, 1975)",0.444000,0.299000,0.299000,0.090000,0.196000,0.000002,0.045706,"23,27"
2,"TMD_C_JMD_C-Pat...4,8)-BEGF750101",Conformation,α-helix,α-helix,"Conformational ...in-Dirkx, 1975)",0.444000,0.299000,0.299000,0.090000,0.196000,0.000002,0.015235,"24,28"
3,"TMD_C_JMD_C-Pat...,12)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.431000,0.251000,-0.251000,0.088000,0.143000,0.000003,0.005935,"29,33,37"
4,"TMD_C_JMD_C-Pat...,12)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.431000,0.251000,-0.251000,0.088000,0.143000,0.000003,0.005564,"24,28,32"
5,"TMD_C_JMD_C-Pat...,12)-MUNV940102",Energy,Free energy (folding),Free energy (α-helix),"Free energy in ...-Serrano, 1994)",0.422000,0.148000,-0.148000,0.056000,0.097000,0.000005,0.003512,"29,33,37"
6,"TMD_C_JMD_C-Pat...,12)-MUNV940102",Energy,Free energy (folding),Free energy (α-helix),"Free energy in ...-Serrano, 1994)",0.422000,0.148000,-0.148000,0.056000,0.097000,0.000005,0.003902,"24,28,32"
7,"TMD_C_JMD_C-Pat...5,8)-ZIMJ680104",Energy,Isoelectric point,Isoelectric point,"Isoelectric poi...n et al., 1968)",0.402000,0.130000,0.130000,0.080000,0.088000,0.000013,0.005435,"33,36,39"
8,"TMD_C_JMD_C-Pat...,14)-ZIMJ680104",Energy,Isoelectric point,Isoelectric point,"Isoelectric poi...n et al., 1968)",0.402000,0.130000,0.130000,0.080000,0.088000,0.000013,0.005359,"28,31,34"
9,"TMD_C_JMD_C-Seg...,11)-AURR980119",Conformation,"α-helix (C-term, out)","α-helix (C-terminal, outside)","Normalized posi...ora-Rose, 1998)",0.392000,0.198000,0.198000,0.106000,0.178000,0.000022,0.005822,"39,40"
10,"TMD_C_JMD_C-Seg...,15)-AURR980119",Conformation,"α-helix (C-term, out)","α-helix (C-terminal, outside)","Normalized posi...ora-Rose, 1998)",0.392000,0.198000,0.198000,0.106000,0.178000,0.000022,0.005769,38


The remaining parameters control how the feature matrix is built or let you reuse one. A custom ``df_scales`` overrides the default scale set; ``accept_gaps`` tolerates gapped parts; ``n_jobs`` parallelizes the build. A pre-computed ``X`` (column ``i`` = feature in row ``i``) skips the build entirely and also covers a :meth:`CPP.run_num` ``df_feat``:

In [3]:
df_scales = aa.load_scales()
df_var_full = sf.prune_by_variance(df_feat=df_feat, df_parts=df_parts, df_scales=df_scales,
                                   threshold=0.005, accept_gaps=True, n_jobs=1)
X = sf.feature_matrix(features=df_feat, df_parts=df_parts)
df_var_X = sf.prune_by_variance(df_feat=df_feat, X=X, threshold=0.005)
print(f"via df_parts+params: {len(df_var_full)} | via pre-computed X: {len(df_var_X)}")

via df_parts+params: 50 | via pre-computed X: 50


**What can go wrong?** A ``threshold`` above every feature's variance would remove all features, so it raises a ``ValueError`` rather than returning an empty table:

In [4]:
try:
    sf.prune_by_variance(df_feat=df_feat, X=X, threshold=1e6)
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: 'threshold' (1000000.0) removed all features. Lower it to retain features.
